# Batch FCFF DCF 실행 (디버깅용 셀 분할 버전)

## 사용 방법

### 전제 조건
1. **V9 DCF 노트북** Cell 1~4 가 먼저 실행되어 있어야 합니다.
   → `DCFModel`, `engine`, `process_one_ticker`, `clear_memory`,
     `US_TICKER_LIST`, `TICKER_START`, `TICKER_END`, `SKIP_DONE`,
     `DONE_PATH`, `FAIL_PATH`, `log` 가 globals 에 존재해야 함
2. **`v9_dcf_excel_export_patch.ipynb`** 의 39개 셀이 실행되어 있어야 합니다.
   → `DCFModel.export_v10_excel` 메서드 주입 필수 (단, `EXPORT_EXCEL=True` 일 때만 사용)

### 실행 흐름
| 단계 | 셀 | 역할 |
|---|---|---|
| Step 1 | 옵션 설정 | `EXPORT_EXCEL` 토글 + 출력 경로 |
| Step 2 | 사전 검증 | 9가지 사전 조건 체크 |
| Step 3 | 체크포인트 로드 | `done_set` 확인 + 잔여 ticker 수 표시 |
| Step 4 | (옵션) Dry-run | 첫 1개 ticker로 환경 검증 (실제 batch 전 권장) |
| Step 5 | **본 실행** | 2,000 ticker 루프 (가장 시간 소요) |
| Step 6 | 결과 요약 | TOP 20 + 통계 |
| Step 7 | (옵션) Fail 분석 | 실패 ticker 별 사유 그룹핑 |
| Step 8 | (옵션) 체크포인트 관리 | done/failed 파일 관리 |

각 단계는 격리되어 있어 **중간에 중단되거나 오류가 나도 partial 결과가 남습니다.**
재시작 시 `SKIP_DONE=True` 가 done 리스트를 자동으로 건너뜁니다.


## Step 1 · 옵션 설정

**여기서만 실험 옵션을 바꾸시면 됩니다.** 다른 셀은 그대로 실행하세요.


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  ★ 배치 옵션 ★
# ═══════════════════════════════════════════════════════════════

# Excel 자동 생성 (★ default False — DB 저장만)
EXPORT_EXCEL = False             # ← True 로 바꾸면 ticker 별 xlsx 생성

# Dry-run 옵션 — 실제 batch 전에 첫 N 개 ticker 만 시험 실행
DRY_RUN_N    = 1                 # ← Step 4 에서 사용 (0 이면 dry-run 스킵)

# ═══════════════════════════════════════════════════════════════

# ── 출력 경로 (자동 분기) ──────────────────────────────────────
import os
from pathlib import Path

EXPORT_DIR_BATCH = (Path(r"C:/reports/batch") if os.name == "nt"
                    else Path.home() / "reports" / "batch")

if EXPORT_EXCEL:
    EXPORT_DIR_BATCH.mkdir(parents=True, exist_ok=True)
    log("BATCH", f"EXPORT_EXCEL=True → 각 ticker 별 Excel 생성: {EXPORT_DIR_BATCH}")
    log("BATCH", f"  ⚠ 디스크 사용량: ~30KB × 2000 ticker ≈ 60MB")
else:
    log("BATCH", "EXPORT_EXCEL=False — DB 저장만 (기본 동작)")

print(f"\n  TICKER_START : {TICKER_START}")
print(f"  TICKER_END   : {TICKER_END}")
print(f"  SKIP_DONE    : {SKIP_DONE}")
print(f"  EXPORT_EXCEL : {EXPORT_EXCEL}")
print(f"  DRY_RUN_N    : {DRY_RUN_N}")


## Step 2 · 사전 조건 검증

V9 노트북의 globals 가 모두 준비됐는지 9가지 항목을 확인합니다.
하나라도 실패하면 `RuntimeError` 가 발생해서 다음 단계로 못 넘어갑니다.


In [ ]:
# ── 사전 조건 검증 ─────────────────────────────────────────────
print("=" * 70)
print("[Step 2] 사전 조건 검증")
print("=" * 70)

_errors = []

# (1) DCFModel
if "DCFModel" in globals():
    print(f"  ✓ DCFModel:           정의됨")
else:
    _errors.append("DCFModel 없음 — V9 DCF 노트북 Cell 4 를 먼저 실행하세요.")

# (2) process_one_ticker (EXPORT_EXCEL=False 일 때 사용)
if "process_one_ticker" in globals():
    print(f"  ✓ process_one_ticker: 정의됨")
else:
    _errors.append("process_one_ticker 없음 — V9 DCF 노트북 Cell 4 끝부분 확인.")

# (3) export_v10_excel (EXPORT_EXCEL=True 일 때만 필요)
if EXPORT_EXCEL:
    if "DCFModel" in globals() and hasattr(DCFModel, "export_v10_excel"):
        print(f"  ✓ export_v10_excel:   메서드 주입됨")
    else:
        _errors.append("export_v10_excel 없음 — v9_dcf_excel_export_patch.ipynb 를 먼저 실행하세요.")
else:
    print(f"  · export_v10_excel:   (EXPORT_EXCEL=False 이므로 미사용)")

# (4) engine
if "engine" in globals():
    print(f"  ✓ DB engine:          연결됨")
else:
    _errors.append("engine 없음 — V9 DCF 노트북 Cell 3 를 먼저 실행하세요.")

# (5) ticker list
if "US_TICKER_LIST" in globals() and len(US_TICKER_LIST) > 0:
    print(f"  ✓ US_TICKER_LIST:     {len(US_TICKER_LIST):,}개")
else:
    _errors.append("US_TICKER_LIST 비어있음 — V9 Cell 2 import 확인.")

# (6) 배치 범위 상수
for name in ["TICKER_START", "TICKER_END", "SKIP_DONE"]:
    if name in globals():
        print(f"  ✓ {name:<20s}: {globals()[name]}")
    else:
        _errors.append(f"{name} 없음 — V9 Cell 2 확인.")

# (7) 체크포인트 경로
for name in ["DONE_PATH", "FAIL_PATH"]:
    if name in globals():
        path = globals()[name]
        parent_writable = os.access(os.path.dirname(os.path.abspath(path)) or ".", os.W_OK)
        if parent_writable:
            print(f"  ✓ {name:<20s}: {path}")
        else:
            _errors.append(f"{name} 의 상위 디렉토리 쓰기 불가: {path}")
    else:
        _errors.append(f"{name} 없음 — V9 Cell 2 확인.")

# (8) clear_memory
if "clear_memory" in globals():
    print(f"  ✓ clear_memory:       정의됨")
else:
    print(f"  ⚠ clear_memory:       없음 — finally 블록에서 스킵됩니다.")

# (9) log
if "log" in globals():
    print(f"  ✓ log:                정의됨")
else:
    _errors.append("log 함수 없음 — V9 Cell 2 확인.")

# 결과
print()
if _errors:
    print("✗ 사전 조건 미충족:")
    for e in _errors:
        print(f"    · {e}")
    raise RuntimeError("Step 2 실패 — 위 메시지 확인")
else:
    print("✓ Step 2 통과")


## Step 3 · 체크포인트 로드

이전 batch 실행에서 완료된 ticker 목록을 확인하고, 잔여 작업량을 계산합니다.
**`SKIP_DONE=True` 일 때만 의미가 있습니다.**


In [ ]:
# ── 체크포인트 로드 ───────────────────────────────────────────
import time
from datetime import datetime

print(f"[Step 3] 체크포인트 로드")

RUN_TICKERS = US_TICKER_LIST[TICKER_START:TICKER_END]
total = len(RUN_TICKERS)
run_date = datetime.now().strftime("%Y-%m-%d")

done_set = set()
if SKIP_DONE and os.path.exists(DONE_PATH):
    with open(DONE_PATH) as f:
        done_set = {l.strip() for l in f if l.strip()}

# 잔여 작업
todo_tickers = [t for t in RUN_TICKERS if t not in done_set]
n_done   = len(RUN_TICKERS) - len(todo_tickers)

print(f"  · 전체 대상 ticker:  {total:,}개  (인덱스 {TICKER_START} ~ {TICKER_END})")
print(f"  · 이미 완료 (skip):  {n_done:,}개")
print(f"  · 처리 예정:         {len(todo_tickers):,}개")
print(f"  · run_date:          {run_date}")

if len(todo_tickers) == 0:
    print(f"\n  ✓ 모든 ticker 가 완료되었습니다. 본 실행 (Step 5) 은 스킵 가능.")
else:
    _est_min = len(todo_tickers) * 1.5 / 60   # 평균 1.5초/ticker 기준
    print(f"\n  · 예상 소요 시간:    ~{_est_min:.1f}분  (평균 1.5초/ticker 기준)")

print(f"\n✓ Step 3 통과")


## Step 4 · (옵션) Dry-run — 첫 N 개로 환경 검증

**실제 batch 실행 (Step 5) 전에 첫 N 개 ticker 로 시험 실행**합니다.
2,000개를 다 돌렸는데 첫 ticker 부터 fail 이 나는 상황을 방지합니다.

`DRY_RUN_N = 0` 이면 이 셀은 스킵됩니다.


In [ ]:
# ── Dry-run ────────────────────────────────────────────────────
if DRY_RUN_N > 0:
    print(f"[Step 4] Dry-run — 첫 {DRY_RUN_N}개 ticker 시험 실행")
    print("=" * 70)

    _dry_targets = todo_tickers[:DRY_RUN_N]
    if not _dry_targets:
        print(f"  ⚠ todo_tickers 가 비어있음 — dry-run 스킵")
    else:
        for _t in _dry_targets:
            print(f"\n  [{_t}] 시험 실행 중 ...")
            _t0 = time.time()

            if EXPORT_EXCEL:
                _model = DCFModel(ticker=_t, engine=engine, verbose=True)
                _model.run()
                _path = _model.export_v10_excel(out_dir=EXPORT_DIR_BATCH)
                print(f"\n    ✓ Excel: {_path}")
                print(f"    · TP=${_model.valuation['target_price']:.2f}")
                print(f"    · WACC={_model.valuation['wacc']*100:.2f}%")
            else:
                _res = process_one_ticker(_t, engine, verbose=True,
                                           run_date=run_date, save_db=False)  # ★ save_db=False (실험)
                if _res["status"] == "ok":
                    print(f"\n    ✓ TP=${_res['target_price']:.2f}  WACC={_res['wacc']*100:.2f}%  (DB 저장 안 됨)")
                else:
                    print(f"\n    ✗ FAIL: {_res['msg']}")

            print(f"    · 시간: {time.time()-_t0:.1f}초")

        print(f"\n✓ Step 4 통과 — Dry-run 성공. Step 5 본 실행 진행 가능.")
        print(f"  ⚠ Dry-run 의 ticker 는 DB 에 저장되지 않았습니다 (save_db=False).")
        print(f"     본 실행에서 다시 처리됩니다.")
else:
    print(f"[Step 4] DRY_RUN_N=0 — Dry-run 스킵")


## Step 5 · 본 실행 (대규모 batch 루프)

가장 시간이 오래 걸리는 단계 — 잔여 ticker 모두를 순차 처리합니다.

### 동작 방식
- `EXPORT_EXCEL=False` (default): `process_one_ticker` 사용 (기존 동작) — 빠름
- `EXPORT_EXCEL=True`: model 인스턴스를 보존해서 `save_to_db` + `export_v10_excel` 둘 다 실행

### 중단 / 재시작
- 셀 실행 중 Ctrl+C / Interrupt 가능 — 그동안 처리된 ticker 는 `DONE_PATH` 에 저장됨
- 재실행 시 `SKIP_DONE=True` 가 자동으로 건너뜀
- 중단 후에도 `results` 리스트에 partial 결과가 남아있어 inspect 가능


In [ ]:
# ── 본 실행 루프 ──────────────────────────────────────────────
print(f"[Step 5] 본 실행 — {len(todo_tickers):,}개 ticker  EXPORT_EXCEL={EXPORT_EXCEL}")
print("=" * 70)

import numpy as np

ok_cnt = skip_cnt = fail_cnt = 0
results = []
t0 = time.time()

log("BATCH", f"배치 시작: {total:,}개  run_date={run_date}  "
             f"SKIP_DONE={SKIP_DONE}  EXPORT_EXCEL={EXPORT_EXCEL}")

for idx, ticker in enumerate(RUN_TICKERS, 1):
    pct    = idx / total * 100
    prefix = f"[{idx:>5}/{total}] ({pct:5.1f}%) {ticker:<8}"

    if SKIP_DONE and ticker in done_set:
        print(f"{prefix} SKIP (checkpoint)", flush=True)
        skip_cnt += 1
        continue

    # ★ EXPORT_EXCEL 분기 ─────────────────────────────────────────
    if EXPORT_EXCEL:
        # 인라인 처리 — model 인스턴스를 보존해서 export_v10_excel 호출
        try:
            model = DCFModel(ticker=ticker, engine=engine, verbose=False)
            model.run()
            rows = model.save_to_db(run_date)

            # Excel 저장은 try/except 로 감싸 — 실패해도 batch 진행
            try:
                model.export_v10_excel(out_dir=EXPORT_DIR_BATCH)
                excel_tag = " +xlsx"
            except Exception as e_x:
                excel_tag = f" (xlsx fail: {str(e_x)[:40]})"

            v = model.valuation
            res = {
                "status":       "ok", "ticker": ticker,
                "target_price": v.get("target_price", np.nan),
                "upside_pct":   v.get("upside_pct", np.nan),
                "wacc":         v.get("wacc", np.nan),
                "g_terminal":   v.get("g_terminal", np.nan),
                "rows_saved":   rows,
                "msg":          f"TP={v.get('target_price', np.nan):.2f}{excel_tag}",
            }
        except Exception as e:
            res = {
                "status": "fail", "ticker": ticker,
                "target_price": np.nan, "upside_pct": np.nan,
                "wacc": np.nan, "g_terminal": np.nan,
                "rows_saved": 0, "msg": str(e)[:120],
            }
        finally:
            if "clear_memory" in globals():
                clear_memory()
    else:
        # 기존 경로 — process_one_ticker 그대로 사용
        res = process_one_ticker(ticker, engine, verbose=False,
                                  run_date=run_date, save_db=True)

    results.append(res)

    if res["status"] == "ok":
        tp  = res["target_price"]
        up  = res["upside_pct"]
        w   = res["wacc"]
        tp_str = f"TP=${tp:.2f}" if not np.isnan(tp) else "TP=N/A"
        up_str = f"↑{up:.1f}%" if not np.isnan(up) else ""
        suffix = res["msg"].split(" ", 1)[1] if " " in res["msg"] and EXPORT_EXCEL else ""
        print(f"{prefix} OK  {tp_str} {up_str}  WACC={w:.3f} {suffix}", flush=True)
        with open(DONE_PATH, "a") as f:
            f.write(ticker + "\n")
        ok_cnt += 1
    else:
        print(f"{prefix} FAIL  {res['msg']}", flush=True)
        with open(FAIL_PATH, "a") as f:
            f.write(ticker + "\n")
        fail_cnt += 1

elapsed = time.time() - t0
print("=" * 70)
log("BATCH", f"완료  OK={ok_cnt}  SKIP={skip_cnt}  FAIL={fail_cnt}  "
             f"경과={elapsed:.0f}s  평균={elapsed/max(ok_cnt+fail_cnt,1):.1f}s/ticker")

print(f"\n✓ Step 5 통과")


## Step 6 · 결과 요약 — TOP 20 + 통계

`results` DataFrame 으로 변환해서 상위 업사이드 종목을 표시합니다.


In [ ]:
# ── 결과 요약 ──────────────────────────────────────────────────
import pandas as pd

print(f"[Step 6] 결과 요약")

if not results:
    print(f"  ⚠ results 가 비어있음 — Step 5 가 실행되지 않았거나 모두 skip 됨")
else:
    summary = pd.DataFrame(results)
    n_ok   = (summary["status"] == "ok").sum()
    n_fail = (summary["status"] == "fail").sum()

    print(f"  · 처리 결과: OK={n_ok}, FAIL={n_fail}")

    if n_ok > 0:
        ok_df = (summary[summary["status"] == "ok"]
                 .sort_values("upside_pct", ascending=False))

        # 통계
        tp_arr  = ok_df["target_price"].dropna()
        up_arr  = ok_df["upside_pct"].dropna()
        w_arr   = ok_df["wacc"].dropna()
        print(f"\n  · TP 분포:        median ${tp_arr.median():.2f},  "
              f"mean ${tp_arr.mean():.2f},  range ${tp_arr.min():.2f}~${tp_arr.max():.2f}")
        print(f"  · Upside 분포:    median {up_arr.median():.1f}%,  "
              f"mean {up_arr.mean():.1f}%,  >50%={(up_arr > 50).sum()}개")
        print(f"  · WACC 분포:      median {w_arr.median()*100:.2f}%,  "
              f"mean {w_arr.mean()*100:.2f}%")

        print(f"\n[상위 업사이드 종목 TOP 20]")
        try:
            display(ok_df[["ticker","target_price","upside_pct","wacc","g_terminal"]].head(20))
        except NameError:
            print(ok_df[["ticker","target_price","upside_pct","wacc","g_terminal"]].head(20).to_string())


## Step 7 · (옵션) Fail 분석

실패한 ticker 들의 사유를 그룹핑해서 패턴을 파악합니다.
같은 사유로 여러 ticker 가 실패하면 코드 수정으로 다수를 한 번에 살릴 수 있습니다.


In [ ]:
# ── Fail 분석 ──────────────────────────────────────────────────
print(f"[Step 7] Fail 분석")

if not results:
    print(f"  ⚠ results 비어있음 — 분석할 fail 없음")
else:
    fail_df = pd.DataFrame(results)
    fail_df = fail_df[fail_df["status"] == "fail"]

    if len(fail_df) == 0:
        print(f"  ✓ 모든 ticker 처리 성공 — fail 없음")
    else:
        # 사유별 그룹핑 (메시지 첫 50자 기준)
        fail_df["msg_key"] = fail_df["msg"].str[:50]
        groups = fail_df.groupby("msg_key").agg(
            count=("ticker", "count"),
            tickers=("ticker", lambda s: ", ".join(s.head(5).tolist())
                                          + (f"... (+{len(s)-5})" if len(s) > 5 else "")),
        ).sort_values("count", ascending=False)

        print(f"  · 총 {len(fail_df)} 개 ticker 실패  ({len(groups)} 가지 사유)")
        print(f"\n[Top 10 fail 사유]")
        try:
            display(groups.head(10))
        except NameError:
            print(groups.head(10).to_string())


## Step 8 · (옵션) 체크포인트 관리

`done_tickers.txt` / `failed_tickers.txt` 파일을 관리합니다. 다음과 같은 경우 사용:

- **다시 실행하고 싶은 ticker가 있을 때**: `done_set` 에서 제거
- **실패한 ticker만 재시도하고 싶을 때**: `failed_tickers.txt` 의 ticker 들을 `done_set` 에서 제거
- **체크포인트 초기화**: 두 파일을 삭제

> **주의**: 이 셀은 파일을 직접 수정합니다. 신중하게 실행하세요.


In [ ]:
# ── 체크포인트 파일 상태 확인 ─────────────────────────────────
print(f"[Step 8] 체크포인트 파일 상태")

for name, path in [("DONE_PATH", DONE_PATH), ("FAIL_PATH", FAIL_PATH)]:
    if os.path.exists(path):
        with open(path) as f:
            tickers = [l.strip() for l in f if l.strip()]
        size_kb = os.path.getsize(path) / 1024
        print(f"  · {name}: {len(tickers):,}개 ticker  ({size_kb:.1f} KB)  → {path}")
    else:
        print(f"  · {name}: 파일 없음  → {path}")

print()
print("─ 사용 예시 (필요 시 주석 해제) ──────────────────────────")
print("# 1. 실패한 ticker 만 재시도하려면:")
print("#    failed_set = set(open(FAIL_PATH).read().split())")
print("#    new_done = [t for t in open(DONE_PATH).read().split() if t not in failed_set]")
print("#    open(DONE_PATH, 'w').write('\\n'.join(new_done))")
print("#    open(FAIL_PATH, 'w').write('')   # fail 리스트 초기화")
print("#")
print("# 2. 체크포인트 완전 초기화:")
print("#    open(DONE_PATH, 'w').close()")
print("#    open(FAIL_PATH, 'w').close()")
print("#")
print("# 3. 특정 ticker만 다시 실행:")
print("#    bad = {'NVDA', 'TSLA'}")
print("#    done = [t for t in open(DONE_PATH).read().split() if t not in bad]")
print("#    open(DONE_PATH, 'w').write('\\n'.join(done))")


## (옵션) Inspection 셀

`results`, `summary`, `fail_df` 등 batch 변수를 자유롭게 inspect 할 수 있습니다.


In [ ]:
# ── 1. 특정 ticker 결과만 추출 ────────────────────────────────
TARGET = "NVDA"   # ← 조회할 ticker
match = [r for r in results if r["ticker"] == TARGET]
if match:
    for k, v in match[0].items():
        print(f"  {k:18s}: {v}")
else:
    print(f"  '{TARGET}' 가 results 에 없습니다 (skip 됐거나 처리 안 됨)")


In [ ]:
# ── 2. WACC 분포 히스토그램 ───────────────────────────────────
import matplotlib.pyplot as plt

if results:
    ok = pd.DataFrame(results)
    ok = ok[ok["status"] == "ok"]
    if len(ok) > 0:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].hist(ok["wacc"].dropna() * 100, bins=30, edgecolor="black", alpha=0.7)
        ax[0].set_xlabel("WACC (%)")
        ax[0].set_ylabel("Frequency")
        ax[0].set_title("WACC 분포")
        ax[0].grid(alpha=0.3)

        up = ok["upside_pct"].dropna()
        up = up[up.between(-100, 300)]   # 극단치 제거
        ax[1].hist(up, bins=40, edgecolor="black", alpha=0.7, color="orange")
        ax[1].axvline(0, color="red", linestyle="--", alpha=0.5)
        ax[1].set_xlabel("Upside (%)")
        ax[1].set_ylabel("Frequency")
        ax[1].set_title("Upside 분포  (-100% ~ +300% 클립)")
        ax[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()


In [ ]:
# ── 3. 생성된 Excel 파일 목록 (EXPORT_EXCEL=True 일 때) ──────
if EXPORT_EXCEL:
    xlsx_files = sorted(EXPORT_DIR_BATCH.glob("*.xlsx"))
    print(f"  · {EXPORT_DIR_BATCH}")
    print(f"  · 총 {len(xlsx_files)}개 Excel 파일")
    if xlsx_files:
        total_kb = sum(f.stat().st_size for f in xlsx_files) / 1024
        print(f"  · 총 크기: {total_kb:.1f} KB ({total_kb/1024:.2f} MB)")
        print(f"\n  · 첫 5개 파일:")
        for f in xlsx_files[:5]:
            print(f"      {f.name}  ({f.stat().st_size / 1024:.1f} KB)")
else:
    print(f"  · EXPORT_EXCEL=False — 생성된 Excel 없음")
